# Construir Gold — KPIs de negócio e observabilidade

Constrói as tabelas Gold a partir da Silver: KPIs de negócio (reconciliação financeira, OTIF, qualidade de produção) e observabilidade (falha cruzada de cadeia fria, qualidade de SKU, estoque negativo).

Diferente da Silver (MERGE por chave única, incremental), a Gold é recriada por completo a cada execução (`overwrite`) — é resultado derivado da Silver, não fonte de verdade própria.

Referências: business-context.md (KPIs documentados), ADR-011 (falha cruzada de cadeia fria).

In [0]:
# gold_reconciliacao_financeira
from pyspark.sql.functions import col, abs as spark_abs

df_pedidos = spark.table("poc_pulse_observability.silver.crm_pedidos")
df_faturas = spark.table("poc_pulse_observability.silver.financeiro_faturas")

df_reconciliacao = (
    df_faturas
    .join(df_pedidos.select("pedido_id", "valor_total"), "pedido_id")
    .withColumn("divergencia_valor", col("valor_faturado") - col("valor_total"))
    .withColumn("divergente", spark_abs(col("divergencia_valor")) > 0.01)
    .select(
        "fatura_id", "pedido_id", "valor_total", "valor_faturado",
        "divergencia_valor", "divergente", "data_faturamento",
    )
)

df_reconciliacao.write.format("delta").mode("overwrite").saveAsTable(
    "poc_pulse_observability.gold.gold_reconciliacao_financeira"
)

total = df_reconciliacao.count()
divergentes = df_reconciliacao.filter(col("divergente")).count()
print(f"Total: {total} | Divergentes: {divergentes} ({divergentes/total:.1%})")

In [0]:
# gold_otif
from pyspark.sql.functions import col, when

df_remessas = spark.table("poc_pulse_observability.silver.tms_remessas")
df_comprovantes = spark.table("poc_pulse_observability.silver.tms_comprovantes_entrega")

df_otif = (
    df_comprovantes
    .join(df_remessas.select("remessa_id", "data_entrega_prevista"), "remessa_id")
    .withColumn(
        "no_prazo",
        when(col("pod_confirmado") == True, col("data_entrega_real") <= col("data_entrega_prevista")).otherwise(None)
    )
    .select(
        "remessa_id", "comprovante_id", "data_entrega_prevista", "data_entrega_real",
        "pod_confirmado", "no_prazo", "status_entrega",
    )
)

df_otif.write.format("delta").mode("overwrite").saveAsTable("poc_pulse_observability.gold.gold_otif")

total = df_otif.count()
confirmadas = df_otif.filter(col("pod_confirmado") == True).count()
no_prazo = df_otif.filter(col("no_prazo") == True).count()
print(f"Total: {total} | Confirmadas: {confirmadas} | No prazo: {no_prazo} ({no_prazo/confirmadas:.1%} das confirmadas)")

In [0]:
# gold_qualidade_producao
from pyspark.sql.functions import col, count, sum as spark_sum, when

df_lotes = spark.table("poc_pulse_observability.silver.erp_lotes_producao")

df_qualidade = (
    df_lotes
    .groupBy("centro_producao_id", "produto_id")
    .agg(
        count("*").alias("total_lotes"),
        spark_sum(when(col("status_qc") == "reprovado", 1).otherwise(0)).alias("lotes_reprovados"),
    )
    .withColumn("taxa_rejeicao", col("lotes_reprovados") / col("total_lotes"))
)

df_qualidade.write.format("delta").mode("overwrite").saveAsTable(
    "poc_pulse_observability.gold.gold_qualidade_producao"
)

df_qualidade.orderBy(col("taxa_rejeicao").desc()).show(20, truncate=False)